# RetailHero — дополнительная проверка устойчивости RFM uplift-модели

**Цель:** оценить устойчивость преимущества uplift-таргетинга на всех доступных размеченных клиентах с помощью **3-fold out-of-fold cross-fitting**.

Что делаем:
1. Фиксируем параметры T-learner без новой настройки по результатам.
2. Трижды обучаем две модели (Control и Treatment) на двух третях данных, получаем прогнозы для оставшейся трети.
3. Для каждого фолда выбираем топ-10% **только по предсказанному uplift** и сравниваем наблюдаемый эффект коммуникации с эффектом случайного выбора в том же фолде.
4. Объединяем оценки по фолдам и строим стратифицированный bootstrap 95% доверительный интервал для **фиксированных моделей и сегментов**.

**Ограничение:** это ретроспективная проверка на том же исходном датасете. RFM-признаки и метод уже были выбраны после просмотра прежнего holdout, поэтому результат не является полностью независимой, заранее зарегистрированной проверкой и не заменяет будущий A/B-тест.

Данные: исходный эксперимент X5 RetailHero и покупки *до* рекламной коммуникации. Точная дата SMS в открытых файлах не приведена.

In [ ]:
%pip -q install huggingface_hub duckdb pandas numpy scikit-learn scipy

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from huggingface_hub import hf_hub_download
from urllib.request import urlretrieve
import subprocess
import sys

SEED = 42
FOLDS = 3
TOP_SHARE = 0.10
N_BOOT = 3000
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
HF_REPO = 'pytorch-lifestream/retailhero-uplift'

def load_file(name):
    path = DATA_DIR / name
    if not path.exists():
        path = Path(hf_hub_download(repo_id=HF_REPO, filename='data/' + name, repo_type='dataset'))
    return pd.read_csv(path, dtype={'client_id': 'string'})

ab = load_file('uplift_train.csv.gz')
clients = load_file('clients.csv.gz')
base = ab.merge(clients, on='client_id', how='left', validate='one_to_one')
assert base.client_id.is_unique and base.treatment_flg.isin([0, 1]).all() and base.target.isin([0, 1]).all()

rfm_path = DATA_DIR / 'final_stats.csv'
if not rfm_path.exists():
    print('Скачиваем и агрегируем официальную историю покупок...')
    purchases = hf_hub_download(repo_id=HF_REPO, filename='data/purchases.csv.gz', repo_type='dataset')
    script_path = Path('build_rfm_features.py')
    urlretrieve(
        'https://raw.githubusercontent.com/venfun77/retailhero-ab-uplift/'
        'd243fbd4adcbf9e3ca33d81620d5387197ff6b87/scripts/build_rfm_features.py',
        script_path
    )
    subprocess.run([sys.executable, str(script_path), '--input', str(purchases),
                    '--output', str(rfm_path)], check=True)

rfm = pd.read_csv(rfm_path, dtype={'client_id': 'string'})
required = ['client_id', 'count', 'sum', 'mean_check', 'recency_days', 'active_span_days',
            'checks_30d', 'checks_90d', 'spending_90d']
assert set(required).issubset(rfm.columns), 'Ожидается полный RFM из скрипта build_rfm_features.py'
assert rfm.client_id.is_unique
rfm = rfm[required].rename(columns={
    'count': 'pre_checks',
    'sum': 'pre_spending',
    'mean_check': 'pre_mean_check',
    'recency_days': 'pre_recency_days',
    'active_span_days': 'pre_active_span_days',
    'checks_30d': 'pre_checks_30d',
    'checks_90d': 'pre_checks_90d',
    'spending_90d': 'pre_spending_90d'
})
data = base.merge(rfm, on='client_id', how='left', validate='one_to_one')
print(f'Обработано {len(data):,} клиентов; история доступна для {data.pre_checks.notna().sum():,}')

## 1. Разбиение и обучение

Тестовый фолд не участвует ни в обучении двух моделей, ни в подборе медиан для заполнения пропусков. Доля Treatment сохранена при разбиении. Используем **те же гиперпараметры T-learner**, что в основном проекте, и не меняем их в зависимости от результатов проверки.

In [ ]:
BEHAVIOR = ['pre_checks', 'pre_spending', 'pre_mean_check', 'pre_recency_days',
            'pre_active_span_days', 'pre_checks_30d', 'pre_checks_90d', 'pre_spending_90d']
NUMERIC_BASE = ['age']

raw = data.copy().reset_index(drop=True)
raw['age'] = pd.to_numeric(raw['age'], errors='coerce').where(
    pd.to_numeric(raw['age'], errors='coerce').between(18, 100))
raw['gender_group'] = raw['gender'].fillna('U').astype(str).where(
    raw['gender'].fillna('U').astype(str).isin(['F','M']), 'U')

y = raw.target.astype(int).to_numpy()
t = raw.treatment_flg.astype(int).to_numpy()
assert np.isin(y, [0,1]).all() and np.isin(t, [0,1]).all()

def build_matrices(train_idx, test_idx, with_rfm):
    cols = NUMERIC_BASE + (BEHAVIOR if with_rfm else [])
    X_train = pd.DataFrame(index=range(len(train_idx)))
    X_test = pd.DataFrame(index=range(len(test_idx)))
    for col in cols:
        series = pd.to_numeric(raw[col], errors='coerce')
        a = series.iloc[train_idx].reset_index(drop=True)
        b = series.iloc[test_idx].reset_index(drop=True)
        median = a.median()
        if pd.isna(median):
            median = 0.
        X_train[col] = a.fillna(median).astype('float32')
        X_test[col] = b.fillna(median).astype('float32')
        X_train[col + '_missing'] = a.isna().astype('int8')
        X_test[col + '_missing'] = b.isna().astype('int8')
    genders = raw.gender_group
    for label in ['F', 'M', 'U']:
        X_train['gender_' + label] = (genders.iloc[train_idx].reset_index(drop=True) == label).astype('int8')
        X_test['gender_' + label] = (genders.iloc[test_idx].reset_index(drop=True) == label).astype('int8')
    return X_train, X_test

def fit_predict_tlearner(train_idx, test_idx, with_rfm):
    X_train, X_test = build_matrices(train_idx, test_idx, with_rfm)
    predictions = {}
    for treatment in [0, 1]:
        mask = t[train_idx] == treatment
        model = HistGradientBoostingClassifier(
            learning_rate=0.05, max_depth=4, max_iter=250,
            min_samples_leaf=80, random_state=SEED)
        model.fit(X_train.loc[mask], y[train_idx][mask])
        predictions[treatment] = model.predict_proba(X_test)[:, 1]
    return predictions[1] - predictions[0]

splitter = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
folds = list(splitter.split(np.arange(len(raw)), t))
all_predictions = []
for fold, (train_idx, test_idx) in enumerate(folds, start=1):
    print(f'Фолд {fold}/{FOLDS}: train={len(train_idx):,}, test={len(test_idx):,}')
    for model_name, with_rfm in [('Демография', False), ('RFM', True)]:
        score = fit_predict_tlearner(train_idx, test_idx, with_rfm)
        result = pd.DataFrame({
            'fold': fold, 'model': model_name, 'treatment': t[test_idx],
            'target': y[test_idx], 'score': score
        })
        all_predictions.append(result)
        print('   ', model_name, '— прогнозы готовы')

predictions = pd.concat(all_predictions, ignore_index=True)
assert predictions.groupby('model').size().eq(len(raw)).all()
print('Каждый клиент оценён out-of-fold в каждой версии модели.')

## 2. Uplift по фолдам и bootstrap

**Оценка эффекта:** разница конверсий Treatment и Control среди клиентов, которых модель включила в топ-10%. **Сравнение:** с ожидаемым эффектом случайной рассылки при той же доле контактов. Доверительные интервалы строятся с повторной генерацией исходов внутри страт фолд × топ/остальные × Treatment/Control и отражают неопределённость исходов при фиксированном ранжировании.

Важно: этот bootstrap не учитывает адаптивный выбор признаков после просмотра результатов прежнего разбиения и не делает новый A/B-тест.

In [ ]:
def calculate_report(frame, seed=42, n_boot=3000):
    frame = frame.copy()
    frame['is_top'] = False
    info = []
    for fold in sorted(frame.fold.unique()):
        idx = frame.index[frame.fold.eq(fold)]
        n_top = max(1, int(len(idx) * TOP_SHARE))
        selected = frame.loc[idx].nlargest(n_top, 'score').index
        frame.loc[selected, 'is_top'] = True
        info.append({'fold': int(fold), 'n_all': len(idx), 'n_top': n_top})

    rng = np.random.default_rng(seed)
    gain_draws = np.zeros(n_boot)
    gain_estimate = 0.0
    uplift_top_estimate = 0.0
    random_estimate = 0.0
    rows = []
    total_size = len(frame)

    for x in info:
        fold, n_all, n_top = x['fold'], x['n_all'], x['n_top']
        part = frame.loc[frame.fold.eq(fold)]
        share = n_top / n_all
        observed = {}
        draws = {}
        for is_top in [False, True]:
            for treatment in [0, 1]:
                z = part.loc[part.is_top.eq(is_top) & part.treatment.eq(treatment), 'target']
                if len(z) == 0:
                    raise ValueError('Пустая страта: нужен другой split')
                rate = z.mean()
                observed[is_top, treatment] = rate
                draws[is_top, treatment] = rng.binomial(len(z), rate, size=n_boot) / len(z)

        tau_top = observed[True, 1] - observed[True, 0]
        tau_rest = observed[False, 1] - observed[False, 0]
        tau_random = share * tau_top + (1-share) * tau_rest
        gain = tau_top - tau_random
        top_draws = draws[True, 1] - draws[True, 0]
        rest_draws = draws[False, 1] - draws[False, 0]
        baseline_draws = share * top_draws + (1-share) * rest_draws
        weight = n_all / total_size
        gain_draws += weight * (top_draws - baseline_draws)
        gain_estimate += weight * gain
        uplift_top_estimate += weight * tau_top
        random_estimate += weight * tau_random
        rows.append({'fold': fold, 'uplift_top_pp':100*tau_top,
                     'random_pp':100*tau_random, 'gain_pp':100*gain})
    low, high = np.quantile(gain_draws, [0.025, 0.975])
    return {
        'top_pp': 100 * uplift_top_estimate,
        'random_pp': 100 * random_estimate,
        'gain_pp': 100 * gain_estimate,
        'ci_low_pp': 100 * low,
        'ci_high_pp': 100 * high,
        'relative_gain_pct': 100 * gain_estimate / random_estimate if random_estimate > 0 else np.nan,
        'folds': pd.DataFrame(rows)
    }

reports = {}
for model_name in ['Демография', 'RFM']:
    results = calculate_report(predictions.loc[predictions.model.eq(model_name)])
    reports[model_name] = results
    print('\n', model_name)
    display(results['folds'].round(3))
    print(f"Топ-10%: {results['top_pp']:+.3f} п.п.; случайный выбор: {results['random_pp']:+.3f} п.п.")
    print(f"Преимущество: {results['gain_pp']:+.3f} п.п., 95% ДИ [{results['ci_low_pp']:+.3f}; {results['ci_high_pp']:+.3f}] п.п.")
    print(f"Относительное преимущество: {results['relative_gain_pct']:+.1f}%")
    print(f"При 6000 SMS: {6000 * results['gain_pp'] / 100:+.0f} дополнительных покупок (точечная оценка)")

summary = pd.DataFrame([
    {k: v for k, v in value.items() if k != 'folds'} | {'model': name}
    for name, value in reports.items()
])
display(summary.set_index('model').round(3))
Path('outputs').mkdir(exist_ok=True)
summary.to_csv('outputs/crossfit_summary.csv', index=False)
print('Сводка сохранена: outputs/crossfit_summary.csv')

## 3. Как интерпретировать результат

- Если CI для RFM полностью выше нуля и результат сохраняется во всех трёх фолдах, это дополнительное **свидетельство устойчивости**, но не новый независимый эксперимент.
- Если хотя бы в части фолдов результат отрицательный или CI включает 0 — не стоит утверждать доказанное преимущество uplift-таргетинга.
- Сравнение RFM с демографической моделью здесь **описательное**; статистический тест разности эффективности моделей в этой версии не проводился.
- Для принятия решения о внедрении всё равно нужен A/B-тест двух политик на новых клиентах.

**Не подбирайте параметры или число фолдов после просмотра результатов, чтобы «добиться значимости».**